# Arabic real estate search: full system prototype

```
Arabic query
   -> normalize digits / amounts (code)
   -> LLM parser -> structured filters (Qwen3)
   -> post-process: canonical districts, amenities, price period (code)
   -> hard filters on the listings table (pandas)  [+ relaxation if nothing matches]
   -> rank the survivors: embeddings (BGE-M3) + keyword overlap
   -> results
```

**The listings here are synthetic** (generated in section 3) so the whole pipeline runs without scraping anyone.
Prices and descriptions are illustrative, not real market data. To plug in real data, replace `listings` with a
DataFrame that has the same columns.

Setup (once):
```
uv add transformers accelerate pydantic pandas numpy torch sentence-transformers
uv add --dev ipykernel
```

In [23]:
import json, re, time, random
from typing import Literal, Optional
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field, ValidationError


import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

LLM_NAME = "Qwen/Qwen3-4B"
EMBEDDING_LLM = "BAAI/bge-m3"
SEED = 42
N_LISTINGS = 600
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)


In [24]:
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, torch_dtype='auto', device_map='auto')
embedder = SentenceTransformer(EMBEDDING_LLM, device=DEVICE)
print("LLM device:", llm.device, "| embedder device:", embedder.device)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

LLM device: cuda:0 | embedder device: cuda:0


## 2. Arabic text utilities (deterministic, no LLM)

Two jobs: normalize spelling so "شقة / شقه" and "اكتوبر / أكتوبر" match, and turn "٥ مليون" into `5000000`
before the LLM sees it. Small models are much more reliable when the arithmetic is already done.

In [25]:
AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩۰۱۲۳۴۵۶۷۸۹", "01234567890123456789")
DIACRITICS = re.compile(r"[\u0617-\u061A\u064B-\u0652\u0640]")   # tashkeel + tatweel


def normalize_ar(text: str) -> str:
    text = text.translate(AR_DIGITS)
    text = DIACRITICS.sub("", text)
    text = re.sub("[إأآ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه")
    return re.sub(r"\s+", " ", text).strip().lower()

def expand_amounts(text: str) -> str:
    """'٥ مليون' -> '5000000', '15 ألف' -> '15000', '2.5 مليون' -> '2500000'."""
    text = text.translate(AR_DIGITS)

    def repl(m):
        num = float(m.group(1))
        unit = m.group(2).lower()
        mult = 1_000_000 if unit in ("مليون", "ملايين") else 1_000
        return str(int(num * mult))

    return re.sub(r"(\d+(?:\.\d+)?)\s*(مليون|ملايين|ألف|الف|k)", repl, text, flags=re.IGNORECASE)


for t in ["شقة للبيع تحت ٥ مليون", "استوديو بحد أقصى ١٥ ألف", "فيلا 2.5 مليون في أكتوبر", "شقه للايجار"]:
    print(t, "->", expand_amounts(t), "|", normalize_ar(t))

شقة للبيع تحت ٥ مليون -> شقة للبيع تحت 5000000 | شقه للبيع تحت 5 مليون
استوديو بحد أقصى ١٥ ألف -> استوديو بحد أقصى 15000 | استوديو بحد اقصي 15 الف
فيلا 2.5 مليون في أكتوبر -> فيلا 2500000 في أكتوبر | فيلا 2.5 مليون في اكتوبر
شقه للايجار -> شقه للايجار | شقه للايجار


## 3. Sample listings (synthetic)

Districts, aliases, amenity vocabulary, and a seeded generator. The alias tables matter on real data too:
"التجمع", "القاهرة الجديدة" and "New Cairo" must all land on one canonical district.

In [26]:
DISTRICTS = {
    "التجمع الخامس": {"city": "القاهرة", "sqm": 55000, "aliases": ["التجمع", "التجمع الخامس", "القاهرة الجديدة", "new cairo", "tagamoa"]},
    "مدينة نصر":     {"city": "القاهرة", "sqm": 38000, "aliases": ["مدينة نصر", "nasr city"]},
    "المعادي":       {"city": "القاهرة", "sqm": 52000, "aliases": ["المعادي", "maadi"]},
    "الزمالك":       {"city": "القاهرة", "sqm": 85000, "aliases": ["الزمالك", "zamalek"]},
    "المهندسين":     {"city": "الجيزة",  "sqm": 55000, "aliases": ["المهندسين", "mohandessin"]},
    "وسط البلد":     {"city": "القاهرة", "sqm": 35000, "aliases": ["وسط البلد", "downtown"]},
    "الرحاب":        {"city": "القاهرة", "sqm": 34000, "aliases": ["الرحاب", "rehab"]},
    "الشروق":        {"city": "القاهرة", "sqm": 24000, "aliases": ["الشروق", "shorouk"]},
    "فيصل":          {"city": "الجيزة",  "sqm": 22000, "aliases": ["فيصل", "faisal"]},
    "الهرم":         {"city": "الجيزة",  "sqm": 24000, "aliases": ["الهرم", "haram"]},
    "الشيخ زايد":    {"city": "الجيزة",  "sqm": 42000, "aliases": ["الشيخ زايد", "زايد", "sheikh zayed"]},
    "6 أكتوبر":      {"city": "الجيزة",  "sqm": 28000, "aliases": ["6 of october", "october", "اكتوبر", "6 اكتوبر"]},
}
ALIAS_TO_DISTRICT = {normalize_ar(a): d for d, info in DISTRICTS.items() for a in info["aliases"]}
CITY_ALIASES = {"القاهره": "القاهرة", "cairo": "القاهرة", "الجيزه": "الجيزة", "giza": "الجيزة"}

AMENITY_ALIASES = {
    "garden":   ["garden", "جنينة", "حديقة"],
    "pool":     ["pool", "حمام سباحة", "بيسين", "سويمنج"],
    "parking":  ["parking", "garage", "جراج", "موقف"],
    "elevator": ["elevator", "lift", "أسانسير", "مصعد"],
    "security": ["security", "أمن", "سيكيورتي"],
    "balcony":  ["balcony", "terrace", "بلكونة", "تراس"],
}
AMENITY_LOOKUP = {normalize_ar(a): k for k, v in AMENITY_ALIASES.items() for a in v}


def canonical_district(name):
    if not name:
        return None
    key = normalize_ar(name)
    if key in ALIAS_TO_DISTRICT:
        return ALIAS_TO_DISTRICT[key]
    for alias, canon in sorted(ALIAS_TO_DISTRICT.items(), key=lambda kv: -len(kv[0])):
        if alias in key:               # e.g. "فالتجمع" contains "التجمع"
            return canon
    return None


def canonical_amenity(name):
    key = normalize_ar(name)
    if key in AMENITY_LOOKUP:
        return AMENITY_LOOKUP[key]
    for alias, canon in AMENITY_LOOKUP.items():
        if alias in key:
            return canon
    return None


print(canonical_district("New Cairo"), "|", canonical_district("فالتجمع"), "|", canonical_district("اكتوبر"), "|", canonical_district("الماظة"))
print(canonical_amenity("بجنينة"), "|", canonical_amenity("pool"), "|", canonical_amenity("اطلالة"))

التجمع الخامس | التجمع الخامس | 6 أكتوبر | None
garden | pool | None


In [27]:
TYPES = {
    "apartment": {"ar": "شقة",         "area": (70, 220),  "w": 0.55, "f": 1.00},
    "villa":     {"ar": "فيلا",        "area": (250, 600), "w": 0.10, "f": 1.15},
    "duplex":    {"ar": "دوبلكس",      "area": (180, 350), "w": 0.10, "f": 1.00},
    "studio":    {"ar": "استوديو",     "area": (35, 65),   "w": 0.08, "f": 1.05},
    "office":    {"ar": "مكتب إداري",  "area": (40, 200),  "w": 0.10, "f": 0.90},
    "shop":      {"ar": "محل",         "area": (25, 120),  "w": 0.07, "f": 1.30},
}
FINISH_AR = {"core_shell": "على الطوب", "semi_finished": "نص تشطيب", "fully_finished": "تشطيب كامل", "super_lux": "سوبر لوكس"}
FINISH_F = {"core_shell": 0.75, "semi_finished": 0.88, "fully_finished": 1.0, "super_lux": 1.15}
AMENITY_AR = {"garden": "جنينة خاصة", "pool": "حمام سباحة", "parking": "جراج", "elevator": "أسانسير", "security": "أمن 24 ساعة", "balcony": "بلكونة واسعة"}
COMPOUND_NAMES = ["الياسمين", "النخيل", "الواحة", "الأندلس", "اللوتس", "الورود", "السلام", "الربيع", "الفردوس", "الزهراء", "النرجس", "الريان"]
METRO_DISTRICTS = {"المعادي", "الهرم", "فيصل", "المهندسين", "وسط البلد"}
VIBES = {
    "quiet":    "في شارع هادي بعيد عن الزحمة",
    "metro":    "قريبة من محطة المترو",
    "schools":  "قريبة من مدارس ونوادي",
    "view":     "إطلالة مفتوحة على حديقة",
    "services": "كل الخدمات حواليها من مولات وصيدليات",
}


def make_listings(n=N_LISTINGS, seed=SEED):
    rng = random.Random(seed)
    names = list(DISTRICTS)
    rows = []
    for i in range(n):
        di = rng.randrange(len(names))
        d = names[di]
        info = DISTRICTS[d]
        t = rng.choices(list(TYPES), weights=[TYPES[k]["w"] for k in TYPES])[0]
        intent = rng.choices(["buy", "rent"], weights=[0.55, 0.45])[0]
        area = rng.randint(*TYPES[t]["area"]) // 5 * 5

        if t == "apartment":
            beds = min(4, max(1, round(area / 55)))
        elif t == "villa":
            beds = rng.randint(4, 7)
        elif t == "duplex":
            beds = rng.randint(3, 5)
        elif t == "studio":
            beds = 1
        else:
            beds = 0
        baths = 1 if beds <= 1 else rng.randint(max(1, beds - 1), beds)

        finishing = rng.choices(list(FINISH_AR), weights=[0.15, 0.20, 0.45, 0.20])[0]
        if intent == "buy":
            delivery = rng.choices(["ready", "resale", "under_construction"], weights=[0.5, 0.3, 0.2])[0]
            payment = "installments" if delivery == "under_construction" else rng.choice(["cash", "installments"])
            furnished = rng.random() < 0.08
        else:
            delivery, payment = "ready", None
            furnished = rng.random() < 0.40

        compound = None
        if rng.random() < 0.45:
            compound = "كمبوند " + COMPOUND_NAMES[(di * 3 + rng.randrange(3)) % len(COMPOUND_NAMES)]

        p = {
            "garden": 0.5 if t in ("villa", "duplex") else 0.12,
            "pool": 0.5 if t == "villa" else (0.25 if compound else 0.03),
            "parking": 0.55,
            "elevator": 0.7 if t in ("apartment", "duplex", "studio", "office") else 0.1,
            "security": 0.75 if compound else 0.25,
            "balcony": 0.6 if t in ("apartment", "duplex", "studio") else 0.1,
        }
        amenities = sorted(a for a, pr in p.items() if rng.random() < pr)

        vibe_pool = [v for v in VIBES if v != "metro" or d in METRO_DISTRICTS]
        vibes = rng.sample(vibe_pool, k=rng.choice([0, 1, 1, 2]))

        total = info["sqm"] * area * TYPES[t]["f"] * FINISH_F[finishing] * rng.lognormvariate(0, 0.12)
        if delivery == "under_construction":
            total *= 0.9
        if intent == "buy":
            price, period = max(50_000, round(total / 50_000) * 50_000), "total"
        else:
            price, period = max(3_000, round(total * 0.0045 / 500) * 500), "monthly"

        kind = TYPES[t]["ar"]
        head = f"{kind} {'للبيع' if intent == 'buy' else 'للإيجار'} في {d}"
        if compound:
            head += f" داخل {compound}"
        parts = [head, f"مساحة {area} متر"]
        if beds == 1:
            parts.append("غرفة نوم واحدة")
        elif beds > 1:
            parts.append(f"{beds} غرف نوم و{baths} حمام")
        parts.append(FINISH_AR[finishing])
        parts += [AMENITY_AR[a] for a in amenities]
        parts += [VIBES[v] for v in vibes]
        if intent == "buy":
            parts.append({"ready": "استلام فوري", "resale": "ريسيل", "under_construction": "تحت الإنشاء استلام بعد سنتين"}[delivery])
            parts.append("كاش" if payment == "cash" else "تقسيط على 8 سنين")
        else:
            parts.append("مفروشة بالكامل" if furnished else "غير مفروشة")

        rows.append({
            "id": i, "intent": intent, "property_type": t, "city": info["city"], "district": d,
            "compound": compound, "bedrooms": beds, "bathrooms": baths, "area_sqm": area,
            "price": int(price), "price_period": period, "payment": payment, "furnished": bool(furnished),
            "finishing": finishing, "delivery": delivery, "amenities": amenities,
            "description": "، ".join(parts) + ".", "url": f"synthetic://listing/{i}",
        })
    return pd.DataFrame(rows)


listings = make_listings()
listings["city_norm"] = listings["city"].map(normalize_ar)
listings["compound_norm"] = listings["compound"].fillna("").map(normalize_ar)
listings["doc"] = listings["description"] + " " + listings["city"]
print(listings.shape)

(600, 21)


In [28]:
display(listings[["id", "intent", "property_type", "district", "bedrooms", "area_sqm", "price", "price_period", "description"]].head(5))
print(listings.intent.value_counts().to_dict())
print(listings.property_type.value_counts().to_dict())
print(listings.groupby("intent").price.describe()[["min", "50%", "max"]])

,id,intent,property_type,district,bedrooms,area_sqm,price,price_period,description
0,0,rent,apartment,الشيخ زايد,2,130,16000,monthly,شقة للإيجار في الشيخ زايد، مساحة 130 متر، 2 غرف نوم و1 حمام، على الطوب، أسانسير، جراج، أمن 24 سا...
1,1,buy,studio,التجمع الخامس,1,45,2050000,total,استوديو للبيع في التجمع الخامس داخل كمبوند النخيل، مساحة 45 متر، غرفة نوم واحدة، نص تشطيب، أمن 2...
2,2,rent,villa,وسط البلد,5,270,58000,monthly,فيلا للإيجار في وسط البلد، مساحة 270 متر، 5 غرف نوم و5 حمام، سوبر لوكس، جنينة خاصة، حمام سباحة، ...
3,3,buy,apartment,6 أكتوبر,2,135,2650000,total,شقة للبيع في 6 أكتوبر داخل كمبوند الزهراء، مساحة 135 متر، 2 غرف نوم و1 حمام، تشطيب كامل، جراج، أ...
4,4,buy,apartment,الشروق,4,210,6200000,total,شقة للبيع في الشروق، مساحة 210 متر، 4 غرف نوم و4 حمام، تشطيب كامل، بلكونة واسعة، أمن 24 ساعة، في...


{'buy': 325, 'rent': 275}
{'apartment': 335, 'duplex': 65, 'office': 61, 'villa': 49, 'shop': 49, 'studio': 41}
             min        50%         max
intent                                 
buy     550000.0  4950000.0  27450000.0
rent      3500.0    22500.0    175000.0


## 4. Filter schema and prompt

Same schema as before plus a `sort` field (for "أرخص"). The prompt states the dialect vocabulary explicitly
(ريسيل, كاش, سوبر لوكس...) because that is where small models guess.

In [29]:
from pydantic import ConfigDict, model_validator

class SearchFilters(BaseModel):
    model_config = ConfigDict(extra="forbid")
    intent: Optional[Literal["buy", "rent"]] = None
    property_type: Optional[Literal["apartment", "villa", "duplex", "studio", "office", "shop"]] = None
    city: Optional[str] = None
    district: Optional[str] = None
    compound: Optional[str] = None
    bedrooms_min: Optional[int] = None
    bedrooms_max: Optional[int] = None
    bathrooms_min: Optional[int] = None
    area_sqm_min: Optional[int] = None
    area_sqm_max: Optional[int] = None
    price_min: Optional[int] = None
    price_max: Optional[int] = None
    price_period: Optional[Literal["total", "monthly"]] = None
    payment: Optional[Literal["cash", "installments"]] = None
    furnished: Optional[bool] = None
    finishing: Optional[Literal["core_shell", "semi_finished", "fully_finished", "super_lux"]] = None
    delivery: Optional[Literal["ready", "under_construction", "resale"]] = None
    amenities: list[str] = Field(default_factory=list)
    excluded_amenities: list[Literal["garden", "pool", "parking", "elevator", "security", "balcony"]] = Field(default_factory=list)
    semantic_query: Optional[str] = None
    sort: Optional[Literal["price_asc", "price_desc", "area_desc"]] = None
    @model_validator(mode="after")
    def validate_constraints(self):
        for name in ("bedrooms_min", "bedrooms_max", "bathrooms_min", "area_sqm_min", "area_sqm_max", "price_min", "price_max"):
            value = getattr(self, name)
            if value is not None and value < 0:
                raise ValueError(f"{name} must be nonnegative")
        for low, high in (("bedrooms_min", "bedrooms_max"), ("area_sqm_min", "area_sqm_max"), ("price_min", "price_max")):
            a, b = getattr(self, low), getattr(self, high)
            if a is not None and b is not None and a > b:
                raise ValueError(f"{low} exceeds {high}")
        if set(self.amenities) & set(self.excluded_amenities):
            raise ValueError("An amenity cannot be both required and excluded")
        if self.intent == "buy" and self.price_period in ("monthly", "yearly"):
            raise ValueError("Purchase installment amounts need a separate payment schema")
        if self.intent == "rent" and self.price_period == "total":
            raise ValueError("Rental price must specify its rental period")
        return self


In [30]:
SYSTEM_PROMPT = """You convert Arabic (Egyptian dialect or formal) and mixed Arabic/English real estate search queries into JSON filters.

Rules:
- Output ONLY one JSON object. No explanation, no markdown.
- Use null for any field the query does not mention. Never guess. Do not set intent unless the query says buy/rent (للبيع / للإيجار / للايجار).
- Amounts are already converted to plain numbers. "تحت X" or "بحد أقصى X" means price_max = X.
- "N غرف" means bedrooms_min = N and bedrooms_max = N.
- Dialect: ريسيل = delivery resale; استلام فوري = delivery ready; كاش = payment cash; تقسيط = payment installments;
  سوبر لوكس = finishing super_lux; متشطبة / تشطيب كامل = fully_finished; نص تشطيب = semi_finished; على الطوب = core_shell;
  مفروش / مفروشة = furnished true; أرخص = sort price_asc; أغلى = sort price_desc.
- Negation matters: غير مفروشة / مش مفروش means furnished false.
- Explicitly forbidden amenities go in excluded_amenities; do not add them to amenities.
- Never infer buy/rent or price_period from the size of a budget.
- Amenity mapping:
  بجنينة / بجنينة خاصة / جنينة / حديقة = garden
  حمام سباحة / بيسين / سويمنج = pool
  جراج / موقف / garage = parking
  أسانسير / مصعد / lift = elevator
  أمن / سيكيورتي = security
  بلكونة / تراس = balcony
- Put every amenity the user explicitly requests in amenities.
- Do not put requested amenities in semantic_query.
- amenities: use only these words: garden, pool, parking, elevator, security, balcony.
- Preferences with no field (quiet, near metro, near schools, view) go in semantic_query, in Arabic.

JSON schema:
{schema}
"""


FEW_SHOT = [
    (
        "شقة للإيجار في مدينة نصر 3 غرف",
        {
            "intent": "rent",
            "property_type": "apartment",
            "district": "مدينة نصر",
            "bedrooms_min": 3,
            "bedrooms_max": 3,
        },
    ),
    (
        "فيلا في الشيخ زايد بجنينة وحمام سباحة وتحت 10000000",
        {
            "property_type": "villa",
            "district": "الشيخ زايد",
            "price_max": 10000000,
            "amenities": ["garden", "pool"],
        },
    ),
    (
        "أرخص شقة في الرحاب",
        {
            "property_type": "apartment",
            "district": "الرحاب",
            "sort": "price_asc",
        },
    ),
    (
        "شقة هادية قريبة من المترو",
        {
            "property_type": "apartment",
            "semantic_query": "هادية قريبة من المترو",
        },
    ),
    (
        "محل للإيجار",
        {
            "intent": "rent",
            "property_type": "shop",
        },
    ),
    (
        "شقة سوبر لوكس ٣ غرف ٢ حمام دور أرضي بجنينة",
        {
            "property_type": "apartment",
            "finishing": "super_lux",
            "bedrooms_min": 3,
            "bathrooms_min": 2,
            "amenities": ["garden"],
        },
    ),
    (
    "شقة ريسيل متشطبة في الرحاب",
    {
        "property_type": "apartment",
        "delivery": "resale",
        "finishing": "fully_finished",
        "district": "الرحاب"
    }
),
]


def build_messages(query, few_shot=FEW_SHOT):
    schema = json.dumps(SearchFilters.model_json_schema(), ensure_ascii=False)
    messages = [{"role": "system", "content": SYSTEM_PROMPT.format(schema=schema)}]
    for q, a in few_shot:
        messages.append({"role": "user", "content": q})
        messages.append({"role": "assistant", "content": json.dumps(a, ensure_ascii=False)})
    messages.append({"role": "user", "content": query})
    return messages

## 5. LLM parser + post-processing

In [31]:
def generate(messages, max_new_tokens=300):
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True,
        enable_thinking=False,    # Qwen3: skip the reasoning trace
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(llm.device)
    output = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)


def extract_json(text):
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = text[:-3].strip()
    result = json.loads(text)
    if not isinstance(result, dict):
        raise ValueError("Expected exactly one JSON object")
    return result


def parse_query(query):
    """Returns (filters_or_None, raw_text, seconds)."""
    t0 = time.time()
    raw = generate(build_messages(query))
    secs = time.time() - t0
    try:
        return SearchFilters(**extract_json(raw)), raw, secs
    except (ValueError, ValidationError):
        return None, raw, secs

In [32]:
def postprocess(f: SearchFilters):
    """Make the LLM output safe to run against the table. Returns (filters, notes)."""
    d = f.model_dump()
    notes, extra_sem = [], []

    if d["district"]:
        canon = canonical_district(d["district"])
        if canon:
            d["district"] = canon
        else:
            raise ValueError(f"Unsupported district: {d['district']}; do not broaden location silently")

    if d["city"]:
        as_district = canonical_district(d["city"])   # e.g. "القاهرة الجديدة" is really a district
        if d["district"]:
            d["city"] = None                           # district already implies the city
        elif as_district:
            d["district"], d["city"] = as_district, None
        elif normalize_ar(d["city"]) in CITY_ALIASES:
            d["city"] = CITY_ALIASES[normalize_ar(d["city"])]
        else:
            extra_sem.append(d["city"])
            d["city"] = None

    amen = []
    for a in d["amenities"]:
        c = canonical_amenity(a)
        if c:
            amen.append(c)
        else:
            notes.append(f"unknown amenity '{a}' -> moved to semantic query")
            extra_sem.append(a)
    d["amenities"] = sorted(set(amen))

    has_budget = d["price_min"] is not None or d["price_max"] is not None
    if has_budget and not d["price_period"]:
        if d["intent"] == "buy":
            d["price_period"] = "total"
        else:
            raise ValueError("Clarify whether the budget is purchase total, monthly rent, or yearly rent")

    if extra_sem:
        d["semantic_query"] = " ".join(x for x in [d["semantic_query"], *extra_sem] if x)
    return SearchFilters(**d), notes


def understand(query):
    """Raw Arabic query -> (filters or None, notes, raw LLM text, LLM seconds)."""
    filters, raw, secs = parse_query(expand_amounts(query))
    if filters is None:
        return None, [], raw, secs
    try:
        filters, notes = postprocess(filters)
    except (ValueError, ValidationError) as exc:
        return None, [str(exc)], raw, secs
    return filters, notes, raw, secs


f, notes, raw, secs = understand("شقة للبيع في القاهرة الجديدة تحت ٥ مليون تقسيط")
print(raw)
print(f.model_dump(exclude_none=True, exclude_defaults=True), notes, f"{secs:.1f}s")

{"intent": "buy", "property_type": "apartment", "city": "القاهرة الجديدة", "price_max": 5000000, "payment": "installments"}
{'intent': 'buy', 'property_type': 'apartment', 'district': 'التجمع الخامس', 'price_max': 5000000, 'price_period': 'total', 'payment': 'installments'} [] 10.1s


## 6. Hard filters (exact constraints) and relaxation

Budget, bedrooms and district are constraints, so they go through plain pandas, never through the embedding model.
If nothing matches, relax the least important filters one at a time and say which were dropped.

In [33]:
def apply_filters(df, f: SearchFilters, skip=()):
    m = pd.Series(True, index=df.index)

    def need(name):
        return getattr(f, name) is not None and name not in skip

    if need("intent"):         m &= df.intent == f.intent
    if need("property_type"):  m &= df.property_type == f.property_type
    if need("city"):           m &= df.city_norm == normalize_ar(f.city)
    if need("district"):       m &= df.district == f.district
    if need("compound"):       m &= df.compound_norm.str.contains(normalize_ar(f.compound), regex=False)
    if need("bedrooms_min"):   m &= df.bedrooms >= f.bedrooms_min
    if need("bedrooms_max"):   m &= df.bedrooms <= f.bedrooms_max
    if need("bathrooms_min"):  m &= df.bathrooms >= f.bathrooms_min
    if need("area_sqm_min"):   m &= df.area_sqm >= f.area_sqm_min
    if need("area_sqm_max"):   m &= df.area_sqm <= f.area_sqm_max
    if need("price_min"):      m &= df.price >= f.price_min
    if need("price_max"):      m &= df.price <= f.price_max
    if need("price_period"):   m &= df.price_period == f.price_period
    if need("payment"):        m &= df.payment == f.payment
    if need("furnished"):      m &= df.furnished == f.furnished
    if need("finishing"):      m &= df.finishing == f.finishing
    if need("delivery"):       m &= df.delivery == f.delivery
    if f.amenities and "amenities" not in skip:
        for a in f.amenities:
            m &= df.amenities.map(lambda s, a=a: a in s)
    for amenity in f.excluded_amenities:
        m &= df.amenities.map(lambda values, a=amenity: a not in values)
    return df[m]


# dropped in this order when a query has no exact match; intent, type and district are never relaxed
RELAX_ORDER = ["finishing", "delivery", "payment", "furnished", "amenities", "bathrooms_min", "area_sqm_max",
               "area_sqm_min", "compound", "price_min", "bedrooms_max", "bedrooms_min", "price_max"]


def search_candidates(df, f, min_results=1, allow_relaxation=False, relax_fields=()):
    """Strict by default. Alternatives require an explicit list of relaxable fields."""
    dropped = []
    cand = apply_filters(df, f)
    protected = {"intent", "property_type", "city", "district", "price_period", "price_max", "excluded_amenities"}
    if allow_relaxation:
        for name in relax_fields:
            if name in protected or name not in RELAX_ORDER:
                raise ValueError(f"Cannot relax {name}")
            if len(cand) >= min_results:
                break
            value = getattr(f, name)
            if value is None or value == []:
                continue
            dropped.append(name)
            cand = apply_filters(df, f, skip=dropped)
    return cand, dropped


test = SearchFilters(intent="buy", property_type="apartment", district="التجمع الخامس", price_max=5_000_000, bedrooms_min=3)
cand, dropped = search_candidates(listings, test)
print(len(cand), "strict matches | relaxed:", dropped)


0 strict matches | relaxed: []


## 7. Ranking: embeddings + keyword overlap

Hard filters decide *who is eligible*; this step decides the *order* (soft preferences like "هادية قريبة من مدارس").
Final score = 0.7 x semantic similarity + 0.3 x keyword overlap, both min-max scaled over the candidates.

In [34]:
def embed_texts(texts, batch_size=32):
    # BGE-M3 needs no prefixes. If you switch to an e5 model, prefix "query: " / "passage: ".
    return np.asarray(embedder.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=False))


STOP = {"في", "من", "علي", "الي", "و", "عايز", "عاوز", "محتاج", "ابغي", "اريد", "لل", "ب"}


def tokenize(text):
    out = set()
    for w in re.findall(r"\w+", normalize_ar(text)):
        for pre in ("وال", "بال", "فال", "لل", "ال"):
            if w.startswith(pre) and len(w) - len(pre) >= 3:
                w = w[len(pre):]
                break
        if len(w) >= 2 and w not in STOP:
            out.add(w)
    return out


t0 = time.time()
listings["tokens"] = listings["doc"].map(tokenize)
doc_emb = embed_texts(listings["doc"].tolist())
print(doc_emb.shape, f"{time.time() - t0:.1f}s")
assert listings.id.is_unique, "Listing IDs must be unique"
EMBED_ROW_BY_ID = {listing_id: pos for pos, listing_id in enumerate(listings.id)}


(600, 1024) 6.5s


In [35]:
def minmax(x):
    x = np.asarray(x, dtype=float)
    rng_ = x.max() - x.min()
    return (x - x.min()) / rng_ if rng_ > 0 else np.zeros_like(x)


def lexical_score(query_tokens, token_sets):
    if not query_tokens:
        return np.zeros(len(token_sets))
    return np.array([len(query_tokens & s) / len(query_tokens) for s in token_sets])


def rank(cand, f, w_sem=0.7, w_lex=0.3):
    if len(cand) == 0:
        return cand
    if f.sort in ("price_asc", "price_desc"):
        if cand[["intent", "price_period"]].drop_duplicates().shape[0] > 1:
            raise ValueError("Choose buy/rent and a comparable price period before sorting by price")
        return cand.sort_values(["price", "id"], ascending=[f.sort == "price_asc", True])
    if f.sort == "area_desc":
        return cand.sort_values("area_sqm", ascending=False)
    if not f.semantic_query:
        return cand.sort_values("id")           # no soft preference: stable order
    q_emb = embed_texts([f.semantic_query])[0]
    positions = [EMBED_ROW_BY_ID[listing_id] for listing_id in cand.id]
    sem = doc_emb[positions] @ q_emb
    lex = lexical_score(tokenize(f.semantic_query), cand["tokens"].tolist())
    out = cand.copy()
    out["semantic_score"] = sem
    out["lexical_score"] = lex
    out["score"] = w_sem * ((sem + 1.0) / 2.0) + w_lex * lex
    return out.sort_values(["score", "id"], ascending=[False, True])


def retrieve(filters, k=5):
    cand, dropped = search_candidates(listings, filters)
    return rank(cand, filters).head(k), dropped, len(cand)


# baselines for the evaluation: no query understanding at all
def keyword_only(query, k=5):
    q_tokens = tokenize(expand_amounts(query))
    score = lexical_score(q_tokens, listings["tokens"].tolist())
    return listings.assign(score=score).sort_values(["score", "id"], ascending=[False, True]).head(k)


def semantic_only(query, k=5):
    sims = doc_emb @ embed_texts([query])[0]
    return listings.assign(score=sims).sort_values("score", ascending=False).head(k)

## 8. End to end

In [36]:
SHOW_COLS = ["id", "intent", "property_type", "district", "bedrooms", "area_sqm", "price", "price_period", "payment", "finishing", "description"]


def show(query, k=5):
    t0 = time.time()
    filters, notes, raw, t_llm = understand(query)
    print("QUERY:", query)
    if filters is None:
        print("No verified results: parsing failed or clarification is required.", notes)
        return listings.iloc[:0].copy()
    print("FILTERS:", filters.model_dump(exclude_none=True, exclude_defaults=True))
    try:
        ranked, dropped, n = retrieve(filters, k)
    except ValueError as exc:
        print("Clarification required:", str(exc))
        return listings.iloc[:0].copy()
    if n == 0:
        print("No exact matches. No constraints were silently removed.")
    print(f"{n} candidates | LLM {t_llm:.1f}s | total {time.time() - t0:.1f}s")
    view = ranked[SHOW_COLS].copy()
    view["description"] = view["description"].str.slice(0, 90)
    display(view)
    return ranked


_ = show("شقة للبيع في التجمع الخامس تحت ٥ مليون تقسيط")

QUERY: شقة للبيع في التجمع الخامس تحت ٥ مليون تقسيط
FILTERS: {'intent': 'buy', 'property_type': 'apartment', 'district': 'التجمع الخامس', 'price_max': 5000000, 'price_period': 'total', 'payment': 'installments'}
2 candidates | LLM 10.2s | total 10.2s


,id,intent,property_type,district,bedrooms,area_sqm,price,price_period,payment,finishing,description
157,157,buy,apartment,التجمع الخامس,2,95,3750000,total,installments,semi_finished,شقة للبيع في التجمع الخامس داخل كمبوند الياسمين، مساحة 95 متر، 2 غرف نوم و1 حمام، نص تشطيب
388,388,buy,apartment,التجمع الخامس,1,75,4550000,total,installments,fully_finished,شقة للبيع في التجمع الخامس، مساحة 75 متر، غرفة نوم واحدة، تشطيب كامل، أسانسير، جراج، أمن 2


In [37]:
for q in [
    "شقة للإيجار في مدينة نصر ٣ غرف",
    "عايز فيلا في الشيخ زايد بجنينة وحمام سباحة",
    "استوديو مفروش في الزمالك بحد أقصى ١٥ ألف في الشهر",
    "شقة هادية في المعادي قريبة من المترو",
    "أرخص شقة في ٦ أكتوبر",
]:
    _ = show(q, k=3)
    print()

QUERY: شقة للإيجار في مدينة نصر ٣ غرف
FILTERS: {'intent': 'rent', 'property_type': 'apartment', 'district': 'مدينة نصر', 'bedrooms_min': 3, 'bedrooms_max': 3}
8 candidates | LLM 9.9s | total 9.9s


,id,intent,property_type,district,bedrooms,area_sqm,price,price_period,payment,finishing,description
16,16,rent,apartment,مدينة نصر,3,170,26500,monthly,None,super_lux,شقة للإيجار في مدينة نصر داخل كمبوند الأندلس، مساحة 170 متر، 3 غرف نوم و2 حمام، سوبر لوكس،
82,82,rent,apartment,مدينة نصر,3,155,27500,monthly,None,super_lux,شقة للإيجار في مدينة نصر، مساحة 155 متر، 3 غرف نوم و2 حمام، سوبر لوكس، بلكونة واسعة، جراج،
271,271,rent,apartment,مدينة نصر,3,145,26000,monthly,None,fully_finished,شقة للإيجار في مدينة نصر، مساحة 145 متر، 3 غرف نوم و3 حمام، تشطيب كامل، بلكونة واسعة، أسان



QUERY: عايز فيلا في الشيخ زايد بجنينة وحمام سباحة
FILTERS: {'property_type': 'villa', 'district': 'الشيخ زايد', 'amenities': ['garden', 'pool']}
2 candidates | LLM 9.2s | total 9.2s


,id,intent,property_type,district,bedrooms,area_sqm,price,price_period,payment,finishing,description
123,123,buy,villa,الشيخ زايد,4,425,14850000,total,cash,core_shell,فيلا للبيع في الشيخ زايد، مساحة 425 متر، 4 غرف نوم و4 حمام، على الطوب، جنينة خاصة، جراج، ح
337,337,rent,villa,الشيخ زايد,5,575,123000,monthly,None,fully_finished,فيلا للإيجار في الشيخ زايد داخل كمبوند السلام، مساحة 575 متر، 5 غرف نوم و4 حمام، تشطيب كام



QUERY: استوديو مفروش في الزمالك بحد أقصى ١٥ ألف في الشهر
FILTERS: {'property_type': 'studio', 'district': 'الزمالك', 'price_max': 15000, 'price_period': 'monthly', 'furnished': True}
No exact matches. No constraints were silently removed.
0 candidates | LLM 9.9s | total 9.9s


,id,intent,property_type,district,bedrooms,area_sqm,price,price_period,payment,finishing,description



QUERY: شقة هادية في المعادي قريبة من المترو
FILTERS: {'property_type': 'apartment', 'district': 'المعادي', 'semantic_query': 'هادية قريبة من المترو'}
26 candidates | LLM 8.9s | total 9.0s


,id,intent,property_type,district,bedrooms,area_sqm,price,price_period,payment,finishing,description
242,242,rent,apartment,المعادي,4,215,52000,monthly,None,fully_finished,شقة للإيجار في المعادي، مساحة 215 متر، 4 غرف نوم و3 حمام، تشطيب كامل، جراج، في شارع هادي ب
396,396,rent,apartment,المعادي,4,195,43000,monthly,None,semi_finished,شقة للإيجار في المعادي، مساحة 195 متر، 4 غرف نوم و4 حمام، نص تشطيب، أسانسير، جراج، أمن 24
99,99,buy,apartment,المعادي,3,145,7450000,total,installments,fully_finished,شقة للبيع في المعادي داخل كمبوند السلام، مساحة 145 متر، 3 غرف نوم و2 حمام، تشطيب كامل، أسا



QUERY: أرخص شقة في ٦ أكتوبر
FILTERS: {'property_type': 'apartment', 'district': '6 أكتوبر', 'sort': 'price_asc'}
Clarification required: Choose buy/rent and a comparable price period before sorting by price



In [38]:
# your own queries here
_ = show("شقه ايجار تحت ال6000")

QUERY: شقه ايجار تحت ال6000
No verified results: parsing failed or clarification is required. ['Clarify whether the budget is purchase total, monthly rent, or yearly rent']


## 9. Evaluation

Two questions:
1. **Parser accuracy:** does it extract the right fields, and does it leave unmentioned fields empty (no hallucination)?
2. **Search quality:** of the top 5 results, how many actually satisfy what the user asked for? Compared with two baselines
   that have no query understanding: plain keyword search and plain embedding search.

The gold set is small on purpose. Grow it to 50-100 queries with real user phrasing, including dialect, typos and mixed
Arabic/English, once you have access to real queries.

In [39]:
GOLD = [
    {"q": "شقة للإيجار في مدينة نصر ٣ غرف",
     "expect": {"intent": "rent", "property_type": "apartment", "district": "مدينة نصر", "bedrooms_min": 3}, "null": ["price_max"]},
    {"q": "عايز فيلا في الشيخ زايد بجنينة وحمام سباحة",
     "expect": {"property_type": "villa", "district": "الشيخ زايد", "amenities": ["garden", "pool"]}, "null": ["intent", "price_max", "bedrooms_min"]},
    {"q": "شقة للبيع في التجمع الخامس تحت ٥ مليون تقسيط",
     "expect": {"intent": "buy", "property_type": "apartment", "district": "التجمع الخامس", "price_max": 5_000_000, "payment": "installments"}, "null": ["bedrooms_min"]},
    {"q": "استوديو مفروش في الزمالك بحد أقصى ١٥ ألف في الشهر",
     "expect": {"property_type": "studio", "district": "الزمالك", "furnished": True, "price_max": 15_000}, "null": ["bedrooms_min", "finishing"]},
    {"q": "شقة هادية في المعادي قريبة من المترو",
     "expect": {"property_type": "apartment", "district": "المعادي"}, "null": ["intent", "price_max", "bedrooms_min"]},
    {"q": "دوبلكس ٢٥٠ متر في كمبوند في أكتوبر",
     "expect": {"property_type": "duplex", "district": "6 أكتوبر"}, "null": ["intent", "price_max"]},
    {"q": "شقة سوبر لوكس ٣ غرف ٢ حمام دور أرضي بجنينة",
     "expect": {"property_type": "apartment", "finishing": "super_lux", "bedrooms_min": 3, "bathrooms_min": 2, "amenities": ["garden"]}, "null": ["intent", "price_max"]},
    {"q": "أرخص شقة في ٦ أكتوبر",
     "expect": {"property_type": "apartment", "district": "6 أكتوبر", "sort": "price_asc"}, "null": ["intent", "bedrooms_min", "price_max"]},
    {"q": "شقة ريسيل متشطبة في الرحاب",
     "expect": {"property_type": "apartment", "delivery": "resale", "finishing": "fully_finished", "district": "الرحاب"}, "null": ["price_max", "bedrooms_min"]},
    {"q": "شقة بين ١٢٠ و١٥٠ متر في فيصل",
     "expect": {"property_type": "apartment", "area_sqm_min": 120, "area_sqm_max": 150, "district": "فيصل"}, "null": ["bedrooms_min", "price_max"]},
    {"q": "شقه للايجار فالتجمع",
     "expect": {"intent": "rent", "property_type": "apartment", "district": "التجمع الخامس"}, "null": ["price_max", "bedrooms_min"]},
    {"q": "شقة",
     "expect": {"property_type": "apartment"}, "null": ["intent", "district", "bedrooms_min", "price_max", "area_sqm_min", "finishing", "payment"]},
]

parsed = {}
rows = []
for g in GOLD:
    filters, notes, raw, secs = understand(g["q"])
    parsed[g["q"]] = filters
    if filters is None:
        rows.append({"query": g["q"], "valid": False, "fields_ok": "0/%d" % len(g["expect"]), "wrong": list(g["expect"]), "hallucinated": [], "secs": round(secs, 1)})
        continue
    got = filters.model_dump()
    wrong = []
    for k, v in g["expect"].items():
        ok = set(v) <= set(got["amenities"]) if k == "amenities" else got.get(k) == v
        if not ok:
            wrong.append(f"{k}: got {got.get(k)!r}")
    halluc = [k for k in g["null"] if got.get(k) is not None]
    rows.append({"query": g["q"], "valid": True, "fields_ok": f"{len(g['expect']) - len(wrong)}/{len(g['expect'])}",
                 "wrong": wrong, "hallucinated": halluc, "secs": round(secs, 1)})

parse_df = pd.DataFrame(rows)
display(parse_df)
total = sum(len(g["expect"]) for g in GOLD)
correct = total - sum(len(r["wrong"]) if r["valid"] else len(GOLD[i]["expect"]) for i, r in enumerate(rows))
print(f"valid JSON: {parse_df.valid.mean():.0%} | field accuracy: {correct}/{total} | queries with hallucinated fields: {sum(bool(h) for h in parse_df.hallucinated)}")

,query,valid,fields_ok,wrong,hallucinated,secs
0,شقة للإيجار في مدينة نصر ٣ غرف,True,4/4,[],[],9.8
1,عايز فيلا في الشيخ زايد بجنينة وحمام سباحة,True,3/3,[],[],9.1
2,شقة للبيع في التجمع الخامس تحت ٥ مليون تقسيط,True,5/5,[],[],10.3
3,استوديو مفروش في الزمالك بحد أقصى ١٥ ألف في الشهر,True,4/4,[],[],9.9
4,شقة هادية في المعادي قريبة من المترو,True,2/2,[],[],9.0
5,دوبلكس ٢٥٠ متر في كمبوند في أكتوبر,True,2/2,[],[],9.6
6,شقة سوبر لوكس ٣ غرف ٢ حمام دور أرضي بجنينة,True,5/5,[],[],10.8
7,أرخص شقة في ٦ أكتوبر,True,3/3,[],[],8.7
8,شقة ريسيل متشطبة في الرحاب,True,4/4,[],[],9.2
9,شقة بين ١٢٠ و١٥٠ متر في فيصل,True,4/4,[],[],9.8


valid JSON: 100% | field accuracy: 40/40 | queries with hallucinated fields: 0


In [40]:
def constraint_precision(results, expect):
    """Share of the returned listings that satisfy every hard constraint the user expressed."""
    if len(results) == 0:
        return 0.0
    hard = {k: v for k, v in expect.items() if k != "sort"}
    ok = apply_filters(results, SearchFilters(**hard))
    return len(ok) / len(results)


rows = []
for g in GOLD:
    f = parsed[g["q"]]
    try:
        system = retrieve(f, k=5)[0] if f is not None else listings.iloc[:0]
    except ValueError:
        system = listings.iloc[:0]
    rows.append({
        "query": g["q"],
        "keyword_only": constraint_precision(keyword_only(g["q"]), g["expect"]),
        "semantic_only": constraint_precision(semantic_only(g["q"]), g["expect"]),
        "full_system": constraint_precision(system, g["expect"]),
    })

cmp_df = pd.DataFrame(rows)
display(cmp_df.round(2))
print(cmp_df[["keyword_only", "semantic_only", "full_system"]].mean().round(2).rename("mean constraint precision@5"))

,query,keyword_only,semantic_only,full_system
0,شقة للإيجار في مدينة نصر ٣ غرف,0.6,1.0,1.0
1,عايز فيلا في الشيخ زايد بجنينة وحمام سباحة,0.4,0.4,1.0
2,شقة للبيع في التجمع الخامس تحت ٥ مليون تقسيط,0.4,0.4,1.0
3,استوديو مفروش في الزمالك بحد أقصى ١٥ ألف في الشهر,0.0,0.0,0.0
4,شقة هادية في المعادي قريبة من المترو,0.8,1.0,1.0
5,دوبلكس ٢٥٠ متر في كمبوند في أكتوبر,0.6,0.8,1.0
6,شقة سوبر لوكس ٣ غرف ٢ حمام دور أرضي بجنينة,0.0,0.0,0.0
7,أرخص شقة في ٦ أكتوبر,1.0,1.0,0.0
8,شقة ريسيل متشطبة في الرحاب,0.0,0.0,0.0
9,شقة بين ١٢٠ و١٥٠ متر في فيصل,0.2,0.2,1.0


keyword_only     0.50
semantic_only    0.57
full_system      0.67
Name: mean constraint precision@5, dtype: float64


## 10. What to do next

1. **Read the failures** in the two evaluation tables before changing anything. Fix the prompt, aliases or schema for the
   pattern you see, then re-run.
2. **Grow the gold set** with real phrasing (friends, family, forums). Add typos, Gulf/Levantine dialect, English-only
   and mixed queries, and vague ones.
3. **Compare models:** change `LLM_ID` only (1.7B / 4B / 8B, a hosted model as the ceiling) and keep the same gold set.
   Do the same with `EMBED_ID`.
4. **Constrained decoding** (e.g. `outlines`, or JSON-schema mode in vLLM/Ollama) to remove invalid JSON.
5. **Real data:** swap `listings` for a sample from a partner or a data feed, keep the same columns, expand the alias tables.
6. **Move to files** once the logic is stable: `src/` modules for text utils, schema, parser, filters, ranking, then a FastAPI endpoint
   the portal can call, and a small RTL demo UI.